# CS 521 HW3, Problem 3: Multi-Norm Adversarial Training

Use Google Colab with a GPU: **Runtime > Change runtime type > T4 GPU**.

All the code is in this notebook. Complete each place that has a `TODO` and `raise NotImplementedError`.
Do not change the data, the model, the attacks, the evaluation or the default hyperparameters,
unless the homework tells you to change them.

**Threat model.** For each test image $x$, the perturbation region is
$B_\infty(x, 8/255) \cup B_2(x, 0.5) \cup B_1(x, 12)$.
The union accuracy is the percentage of test images that are robust against all three attacks simultaneously.

# Set up for dataset and model

Download the base model, a subset of 10,000 CIFAR-10 training images (1,000 for each class),
and the first 1,000 CIFAR-10 test images from Google Drive.

In [1]:
!gdown 17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes -O linf_base.pt
!gdown 1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe -O cifar10_train_10k.pt
!gdown 1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA -O cifar10_test_1k.pt

Downloading...
From: https://drive.google.com/uc?id=17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes
To: /content/linf_base.pt
100% 11.2M/11.2M [00:00<00:00, 121MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe
From (redirected): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe&confirm=t&uuid=9b3e3cfd-43b0-41e5-9d0c-59e097827c84
To: /content/cifar10_train_10k.pt
100% 30.8M/30.8M [00:00<00:00, 141MB/s]
Downloading...
From: https://drive.google.com/uc?id=1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA
To: /content/cifar10_test_1k.pt
100% 3.08M/3.08M [00:00<00:00, 132MB/s]


In [2]:
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

NORMS = ("linf", "l2", "l1")
EPS = {"linf": 8 / 255, "l2": 0.5, "l1": 12.0}

device: cuda


In [3]:
def load_data(path):
    # The files store images as uint8. Convert them to floats in [0, 1] on the GPU.
    x, y = torch.load(path)
    return x.float().div(255).to(device), y.to(device)


# Training data: 10,000 images. Test data: the first 1,000 CIFAR-10 test images.
x_train, y_train = load_data("cifar10_train_10k.pt")
x_test, y_test = load_data("cifar10_test_1k.pt")
print("train:", tuple(x_train.shape), "test:", tuple(x_test.shape))


def augment(x):
    # Random horizontal flip and random crop (padding 4) for each image.
    n = x.shape[0]
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    padded = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 32, 1)
    j = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 1, 32)
    return padded[torch.arange(n, device=x.device).view(n, 1, 1), :, i, j].permute(0, 3, 1, 2).contiguous()


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for k in range(0, len(x), batch_size):
        j = idx[k:k + batch_size]
        yield x[j], y[j]

train: (10000, 3, 32, 32) test: (1000, 3, 32, 32)


In [4]:
# PreActResNet-18 with half width (2.8M parameters). The input is an image in [0, 1].
class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.shortcut = None
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if self.shortcut is not None else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut


class PreActResNet18(nn.Module):
    def __init__(self, width=32, num_classes=10):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.2471, 0.2435, 0.2616]).view(1, 3, 1, 1))
        self.conv1 = nn.Conv2d(3, width, 3, 1, 1, bias=False)
        layers, in_planes = [], width
        for mult, stride in [(1, 1), (1, 1), (2, 2), (2, 1), (4, 2), (4, 1), (8, 2), (8, 1)]:
            layers.append(PreActBlock(in_planes, mult * width, stride))
            in_planes = mult * width
        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(in_planes)
        self.linear = nn.Linear(in_planes, num_classes)

    def forward(self, x):
        out = self.layers(self.conv1((x - self.mean) / self.std))
        out = F.adaptive_avg_pool2d(F.relu(self.bn(out)), 1).flatten(1)
        return self.linear(out)


def make_model():
    return PreActResNet18().to(device)


def load_base_model():
    model = make_model()
    model.load_state_dict(torch.load("linf_base.pt", map_location=device))
    return model

# Implement the Attacks

Each attack is untargeted, uses the cross-entropy loss, and returns `x_adv` with
$\|x_{adv} - x\|_p \leq \epsilon$ and $x_{adv} \in [0, 1]$. The attacks must not change the mode (`train` or `eval`) of the model.

* `pgd_linf`: start at a uniformly random point in the $\ell_\infty$ ball. For each step, add `alpha * sign(grad)`,
  clip the perturbation to $[-\epsilon, \epsilon]$, and clip the image to $[0, 1]$.
* `pgd_l2`: start at a random point in the $\ell_2$ ball. For each step, add `alpha * grad / ||grad||_2`,
  scale the perturbation back to the $\ell_2$ ball if its norm is larger than $\epsilon$, and clip the image to $[0, 1]$.

You can use your code from HW1. The $\ell_1$ attack (sparse $\ell_1$ descent, Tramer & Boneh 2019) is complete.

In [5]:
def clip_box(x, delta):
    # Make x + delta an image in [0, 1]. This never increases |delta_i|.
    return (x + delta).clamp(0, 1) - x


def input_grad(model, x, delta, y):
    delta = delta.detach().requires_grad_(True)
    loss = F.cross_entropy(model(x + delta), y, reduction="sum")
    (grad,) = torch.autograd.grad(loss, delta)
    return grad.detach()


def l2_norm(v):
    return v.flatten(1).norm(dim=1).view(-1, 1, 1, 1)


def pgd_linf(model, x, y, eps, alpha, steps, rand_init=True):
    delta = (torch.rand_like(x) * 2 - 1) * eps if rand_init else torch.zeros_like(x)
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        delta = delta + alpha * grad.sign()
        delta = delta.clamp(-eps, eps)
        delta = clip_box(x, delta)
    return (x + delta).detach()


def pgd_l2(model, x, y, eps, alpha, steps, rand_init=True):
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.randn_like(x)
        delta = delta / l2_norm(delta).clamp_min(1e-12) * eps * torch.rand(x.shape[0], 1, 1, 1, device=x.device)
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        delta = delta + alpha * grad / (l2_norm(grad) + 1e-12)
        delta = delta * (eps / l2_norm(delta).clamp_min(eps))
        delta = clip_box(x, delta)
    return (x + delta).detach()

In [6]:
def project_l1(delta, eps):
    # Projection of each image of delta onto the L1 ball of radius eps (Duchi et al., 2008).
    v = delta.flatten(1)
    a = v.abs()
    outside = a.sum(dim=1) > eps
    if not outside.any():
        return delta
    mu, _ = a.sort(dim=1, descending=True)
    cumsum = mu.cumsum(dim=1)
    idx = torch.arange(1, v.shape[1] + 1, device=v.device, dtype=v.dtype)
    rho = ((mu - (cumsum - eps) / idx > 0) * idx).argmax(dim=1, keepdim=True)
    theta = (cumsum.gather(1, rho) - eps) / (rho + 1).to(v.dtype)
    w = (a - theta).clamp_min(0) * v.sign()
    return torch.where(outside.unsqueeze(1), w, v).view(delta.shape)


def pgd_l1(model, x, y, eps, alpha, steps, rand_init=True, k=20):
    # Each step changes only the k coordinates with the largest gradient.
    # Coordinates at the image boundary that point outward are skipped.
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.distributions.Laplace(0.0, 1.0).sample(x.shape).to(x.device)
        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)
        delta = delta / delta.flatten(1).abs().sum(1).view(-1, 1, 1, 1) * eps * r
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y).flatten(1)
        x_cur = (x + delta).flatten(1)
        blocked = ((grad > 0) & (x_cur >= 1)) | ((grad < 0) & (x_cur <= 0))
        grad = grad.masked_fill(blocked, 0)
        top = grad.abs().topk(k, dim=1).indices
        step = torch.zeros_like(grad).scatter(1, top, grad.gather(1, top).sign())
        delta = clip_box(x, delta + alpha * (step / k).view_as(x))
        delta = project_l1(delta, eps)
    return (x + delta).detach()


ATTACKS = {"linf": pgd_linf, "l2": pgd_l2, "l1": pgd_l1}


def attack(model, x, y, norm, steps=10):
    # The attack for one norm with the default eps and step size.
    alpha = EPS["l1"] / steps ** 0.5 if norm == "l1" else 2.5 * EPS[norm] / steps
    return ATTACKS[norm](model, x, y, EPS[norm], alpha, steps)

# Evaluate Single and Multi-Norm Robust Accuracy

`test_model` gives the clean accuracy, the robust accuracy for each norm, and the union accuracy (in %) on the
1,000 test images. An image is robust for a norm if the model is correct on the clean image and on the result
of each restart of the attack. An image is robust for the union if it is robust for all three norms simultaneously.
Use the default settings (`steps=50, restarts=2`) for all the numbers in your report.

In [7]:
def test_model(model, steps=50, restarts=2, batch_size=250):
    model.eval()
    torch.manual_seed(0)  # the same random starts of the attacks for each model, so the results are comparable
    start = time.time()
    correct = {k: 0 for k in ("clean", *NORMS, "union")}
    for i in range(0, len(x_test), batch_size):
        x, y = x_test[i:i + batch_size], y_test[i:i + batch_size]
        with torch.no_grad():
            clean = model(x).argmax(1) == y
        union = clean.clone()
        for norm in NORMS:
            robust = clean.clone()
            for _ in range(restarts):
                x_adv = attack(model, x, y, norm, steps)
                with torch.no_grad():
                    robust &= model(x_adv).argmax(1) == y
            correct[norm] += robust.sum().item()
            union &= robust
        correct["clean"] += clean.sum().item()
        correct["union"] += union.sum().item()
    acc = {k: 100.0 * v / len(y_test) for k, v in correct.items()}
    print("  ".join(f"{k} {v:.1f}" for k, v in acc.items()) + f"   ({time.time() - start:.0f} s)")
    return acc

A small registry that trains a model from the base model, times it, evaluates it with `test_model`, and stores the result.

In [8]:
import json

import pandas as pd

COLS = ["clean", "linf", "l2", "l1", "union", "sec_per_epoch"]
results = {}
models = {}
histories = {}


def run(name, train_fn, epochs=10):
    model = load_base_model()
    start = time.time()
    out = train_fn(model)
    seconds = time.time() - start
    history = None
    if isinstance(out, tuple):
        model, history = out
    acc = test_model(model)
    acc["sec_per_epoch"] = seconds / epochs
    results[name] = acc
    models[name] = model
    if history is not None:
        histories[name] = history
    json.dump(results, open("results.json", "w"), indent=1)
    return acc


def show(names):
    display(pd.DataFrame({n: results[n] for n in names}).T[COLS].round(1))


# Base Model

In [9]:
# The base model: L-infinity PGD adversarial training for 30 epochs on all 50,000 training images.
results["base"] = test_model(load_base_model())

clean 78.7  linf 48.4  l2 59.8  l1 22.4  union 22.3   (122 s)


# Adversarial Training

`adv_train(model, loss_fn, ...)` fine-tunes `model` for `epochs` epochs on the training subset.
For each batch, it calls `loss_fn(model, x, y, **kwargs)` and does one optimizer step on the loss.

A loss function makes the adversarial examples itself. Put the model in `eval` mode for the attack
(so that the attack does not change the BatchNorm statistics), and in `train` mode before you calculate the loss.

If `gp_beta` is set, `adv_train` does the gradient projection of RAMP (part (b)) before the optimizer step.

In [10]:
def adv_train(model, loss_fn, epochs=10, lr=0.01, batch_size=128, gp_beta=None, seed=0, **kwargs):
    torch.manual_seed(seed)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * (len(x_train) // batch_size))
    params = list(model.parameters())
    start = time.time()
    for epoch in range(epochs):
        losses = []
        for x, y in batches(x_train, y_train, batch_size, shuffle=True):
            if len(y) < batch_size:  # skip the last, smaller batch
                continue
            x = augment(x)
            loss = loss_fn(model, x, y, **kwargs)
            opt.zero_grad()
            if gp_beta is None:
                loss.backward()
            else:
                # RAMP (part (b)): combine the adversarial gradient g_a and the natural gradient g_n
                # for each parameter tensor. gradient_projection is defined in part (b).
                g_a = torch.autograd.grad(loss, params)
                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)
                for p, ga, gn in zip(params, g_a, g_n):
                    p.grad = gp_beta * gradient_projection(gn, ga) + (1 - gp_beta) * ga
            opt.step()
            sched.step()
            losses.append(loss.item())
        print(f"epoch {epoch + 1}: loss {np.mean(losses):.3f} ({time.time() - start:.0f} s)")
    model.eval()
    return model

# Part (a): Single-Norm Adversarial Training

Complete `single_norm_loss`: attack the batch with the norm `norm` (use `attack`), and return the
cross-entropy loss of the model on the adversarial images.

In [11]:
def single_norm_loss(model, x, y, norm):
    model.eval()
    x_adv = attack(model, x, y, norm)
    model.train()
    return F.cross_entropy(model(x_adv), y)

Fine-tune one model per norm.

In [12]:
run("linf", lambda m: adv_train(m, single_norm_loss, norm="linf"))

epoch 1: loss 1.203 (41 s)
epoch 2: loss 1.181 (82 s)
epoch 3: loss 1.151 (122 s)
epoch 4: loss 1.126 (163 s)
epoch 5: loss 1.097 (204 s)
epoch 6: loss 1.068 (244 s)
epoch 7: loss 1.049 (285 s)
epoch 8: loss 1.030 (326 s)
epoch 9: loss 1.013 (366 s)
epoch 10: loss 1.007 (407 s)
clean 78.4  linf 44.8  l2 56.9  l1 19.9  union 19.7   (123 s)


{'clean': 78.4,
 'linf': 44.8,
 'l2': 56.9,
 'l1': 19.9,
 'union': 19.7,
 'sec_per_epoch': 41.193476462364195}

In [13]:
run("l2", lambda m: adv_train(m, single_norm_loss, norm="l2"))

epoch 1: loss 0.897 (41 s)
epoch 2: loss 0.826 (81 s)
epoch 3: loss 0.787 (122 s)
epoch 4: loss 0.754 (162 s)
epoch 5: loss 0.714 (203 s)
epoch 6: loss 0.675 (244 s)
epoch 7: loss 0.631 (284 s)
epoch 8: loss 0.610 (325 s)
epoch 9: loss 0.591 (366 s)
epoch 10: loss 0.584 (406 s)
clean 82.8  linf 32.4  l2 60.0  l1 28.2  union 24.8   (123 s)


{'clean': 82.8,
 'linf': 32.4,
 'l2': 60.0,
 'l1': 28.2,
 'union': 24.8,
 'sec_per_epoch': 40.64446597099304}

In [14]:
run("l1", lambda m: adv_train(m, single_norm_loss, norm="l1"))

epoch 1: loss 1.232 (41 s)
epoch 2: loss 0.824 (82 s)
epoch 3: loss 0.585 (124 s)
epoch 4: loss 0.435 (165 s)
epoch 5: loss 0.341 (206 s)
epoch 6: loss 0.265 (248 s)
epoch 7: loss 0.226 (289 s)
epoch 8: loss 0.172 (330 s)
epoch 9: loss 0.147 (371 s)
epoch 10: loss 0.149 (412 s)
clean 83.5  linf 0.0  l2 0.0  l1 24.7  union 0.0   (123 s)


{'clean': 83.5,
 'linf': 0.0,
 'l2': 0.0,
 'l1': 24.7,
 'union': 0.0,
 'sec_per_epoch': 41.247479510307315}

Part (a) table: the base model and the three single-norm models.

In [15]:
show(["base", "linf", "l2", "l1"])

,clean,linf,l2,l1,union,sec_per_epoch
base,78.7,48.4,59.8,22.4,22.3,NaN
linf,78.4,44.8,56.9,19.9,19.7,41.2
l2,82.8,32.4,60.0,28.2,24.8,40.6
l1,83.5,0.0,0.0,24.7,0.0,41.2


# Part (b): Multi-Norm Adversarial Training (RAMP)

* `ramp_loss`: attack with $q = \ell_\infty$ and $r = \ell_1$. The loss is the mean of
  $\max(\mathcal{L}_{CE}(x^q), \mathcal{L}_{CE}(x^r))$ plus `lam` times $\mathcal{L}_{KL} = KL(p_q \| p_r)$.
  Calculate $\mathcal{L}_{KL}$ only on the images that the model classifies correctly under the $\ell_q$ attack.
* `gradient_projection(g_n, g_a)`: $\cos(g_n, g_a) \cdot g_n$ if $\cos(g_n, g_a) > 0$, else $0$.

`adv_train(..., gp_beta=0.5)` then uses `p.grad = gp_beta * GP(g_n, g_a) + (1 - gp_beta) * g_a` for each parameter
tensor, where $g_a$ is the gradient of `ramp_loss` and $g_n$ is the gradient of the cross-entropy loss on the clean images.

In [16]:
def ramp_loss(model, x, y, lam=0.5):
    model.eval()
    x_q = attack(model, x, y, "linf")
    x_r = attack(model, x, y, "l1")
    model.train()
    logits_q = model(x_q)
    logits_r = model(x_r)
    ce_q = F.cross_entropy(logits_q, y, reduction="none")
    ce_r = F.cross_entropy(logits_r, y, reduction="none")
    loss = torch.maximum(ce_q, ce_r).mean()
    correct = logits_q.argmax(1) == y
    if correct.any():
        log_p_q = F.log_softmax(logits_q[correct], dim=1)
        log_p_r = F.log_softmax(logits_r[correct], dim=1)
        kl = (log_p_q.exp() * (log_p_q - log_p_r)).sum(dim=1).mean()
        loss = loss + lam * kl
    return loss


def gradient_projection(g_n, g_a):
    cos = F.cosine_similarity(g_n.flatten(), g_a.flatten(), dim=0, eps=1e-12)
    return cos.clamp_min(0) * g_n

RAMP with `seed=0`.

In [17]:
run("ramp_s0", lambda m: adv_train(m, ramp_loss, gp_beta=0.5, seed=0))

epoch 1: loss 1.432 (87 s)
epoch 2: loss 1.331 (174 s)
epoch 3: loss 1.303 (261 s)
epoch 4: loss 1.281 (349 s)
epoch 5: loss 1.253 (436 s)
epoch 6: loss 1.227 (524 s)
epoch 7: loss 1.198 (611 s)
epoch 8: loss 1.186 (698 s)
epoch 9: loss 1.171 (785 s)
epoch 10: loss 1.165 (872 s)
clean 79.2  linf 40.7  l2 61.6  l1 44.7  union 39.1   (122 s)


{'clean': 79.2,
 'linf': 40.7,
 'l2': 61.6,
 'l1': 44.7,
 'union': 39.1,
 'sec_per_epoch': 87.23886184692383}

Part (b) table: base, single-norm models and RAMP, with the training time of one epoch.

In [18]:
show(["base", "linf", "l2", "l1", "ramp_s0"])

,clean,linf,l2,l1,union,sec_per_epoch
base,78.7,48.4,59.8,22.4,22.3,NaN
linf,78.4,44.8,56.9,19.9,19.7,41.2
l2,82.8,32.4,60.0,28.2,24.8,40.6
l1,83.5,0.0,0.0,24.7,0.0,41.2
ramp_s0,79.2,40.7,61.6,44.7,39.1,87.2


# Part (c): Your Method

Write the training loss of your method in `my_loss`, and train with `adv_train(load_base_model(), my_loss)`.
If your method needs a different update (not the gradient of one loss), write your own copy of `adv_train`.
Run your method and RAMP with `seed=0` and `seed=1`.
Then evaluate your best model again with stronger attacks: `test_model(model, steps=200, restarts=5)`.

**Method: weakest-norm-first gradient surgery.** Attack with $\ell_\infty$ and $\ell_1$, take one gradient per norm, and remove from the gradient of the stronger norm the part that conflicts with the gradient of the weaker norm.

In [19]:
def my_loss(model, x, y):
    model.eval()
    x_linf = attack(model, x, y, "linf")
    x_l1 = attack(model, x, y, "l1")
    model.train()
    return F.cross_entropy(model(x_linf), y), F.cross_entropy(model(x_l1), y)

`adv_train_wnf` is a copy of `adv_train` with the same hyperparameters and a different update. `mode` is `wnf` (the method), `sum` (no surgery) or `pcgrad` (symmetric surgery).

In [20]:
def flat_grad(loss, params):
    return torch.cat([g.reshape(-1) for g in torch.autograd.grad(loss, params)])


def set_grad(params, flat):
    offset = 0
    for p in params:
        n = p.numel()
        p.grad = flat[offset:offset + n].view_as(p).clone()
        offset += n


def adv_train_wnf(model, loss_fn, epochs=10, lr=0.01, batch_size=128, seed=0, mode="wnf"):
    torch.manual_seed(seed)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * (len(x_train) // batch_size))
    params = list(model.parameters())
    history = []
    start = time.time()
    for epoch in range(epochs):
        stats = {"loss_linf": [], "loss_l1": [], "linf_weak": [], "cos": [], "conflict": []}
        for x, y in batches(x_train, y_train, batch_size, shuffle=True):
            if len(y) < batch_size:
                continue
            x = augment(x)
            loss_linf, loss_l1 = loss_fn(model, x, y)
            g_linf = flat_grad(loss_linf, params)
            g_l1 = flat_grad(loss_l1, params)
            linf_weak = loss_linf.item() >= loss_l1.item()
            weak, strong = (g_linf, g_l1) if linf_weak else (g_l1, g_linf)
            dot = torch.dot(weak, strong)
            cos = dot / (weak.norm() * strong.norm() + 1e-12)
            conflict = dot.item() < 0
            if mode == "wnf" and conflict:
                strong = strong - dot / (weak.dot(weak) + 1e-12) * weak
            elif mode == "pcgrad" and conflict:
                weak, strong = (weak - dot / (strong.dot(strong) + 1e-12) * strong,
                                strong - dot / (weak.dot(weak) + 1e-12) * weak)
            opt.zero_grad()
            set_grad(params, 0.5 * (weak + strong))
            opt.step()
            sched.step()
            stats["loss_linf"].append(loss_linf.item())
            stats["loss_l1"].append(loss_l1.item())
            stats["linf_weak"].append(float(linf_weak))
            stats["cos"].append(cos.item())
            stats["conflict"].append(float(conflict))
        row = {k: float(np.mean(v)) for k, v in stats.items()}
        row["seconds"] = time.time() - start
        history.append(row)
        print(f"epoch {epoch + 1}: loss linf {row['loss_linf']:.3f} l1 {row['loss_l1']:.3f}  "
              f"cos {row['cos']:.3f}  conflict {100 * row['conflict']:.0f}%  linf weaker {100 * row['linf_weak']:.0f}%  "
              f"({row['seconds']:.0f} s)")
    model.eval()
    return model, history

RAMP with `seed=1`, then my method with `seed=0` and `seed=1`.

In [21]:
run("ramp_s1", lambda m: adv_train(m, ramp_loss, gp_beta=0.5, seed=1))

epoch 1: loss 1.434 (87 s)
epoch 2: loss 1.332 (174 s)
epoch 3: loss 1.299 (261 s)
epoch 4: loss 1.274 (348 s)
epoch 5: loss 1.248 (435 s)
epoch 6: loss 1.221 (522 s)
epoch 7: loss 1.200 (609 s)
epoch 8: loss 1.186 (697 s)
epoch 9: loss 1.178 (784 s)
epoch 10: loss 1.169 (871 s)
clean 79.0  linf 42.1  l2 62.7  l1 46.8  union 40.6   (122 s)


{'clean': 79.0,
 'linf': 42.1,
 'l2': 62.7,
 'l1': 46.8,
 'union': 40.6,
 'sec_per_epoch': 87.12714717388153}

In [22]:
run("mine_s0", lambda m: adv_train_wnf(m, my_loss, seed=0))

epoch 1: loss linf 1.287 l1 1.314  cos 0.778  conflict 0%  linf weaker 67%  (82 s)
epoch 2: loss linf 1.259 l1 1.156  cos 0.796  conflict 0%  linf weaker 100%  (164 s)
epoch 3: loss linf 1.242 l1 1.108  cos 0.783  conflict 0%  linf weaker 100%  (245 s)
epoch 4: loss linf 1.221 l1 1.066  cos 0.765  conflict 0%  linf weaker 100%  (327 s)
epoch 5: loss linf 1.193 l1 1.029  cos 0.754  conflict 0%  linf weaker 100%  (408 s)
epoch 6: loss linf 1.165 l1 0.993  cos 0.744  conflict 0%  linf weaker 100%  (490 s)
epoch 7: loss linf 1.137 l1 0.957  cos 0.735  conflict 0%  linf weaker 100%  (572 s)
epoch 8: loss linf 1.119 l1 0.938  cos 0.726  conflict 0%  linf weaker 100%  (653 s)
epoch 9: loss linf 1.103 l1 0.922  cos 0.732  conflict 0%  linf weaker 100%  (735 s)
epoch 10: loss linf 1.098 l1 0.917  cos 0.723  conflict 0%  linf weaker 100%  (817 s)
clean 77.9  linf 40.7  l2 61.8  l1 48.2  union 39.4   (123 s)


{'clean': 77.9,
 'linf': 40.7,
 'l2': 61.8,
 'l1': 48.2,
 'union': 39.4,
 'sec_per_epoch': 81.70235798358917}

In [23]:
run("mine_s1", lambda m: adv_train_wnf(m, my_loss, seed=1))

epoch 1: loss linf 1.284 l1 1.316  cos 0.776  conflict 0%  linf weaker 65%  (82 s)
epoch 2: loss linf 1.262 l1 1.161  cos 0.791  conflict 0%  linf weaker 100%  (164 s)
epoch 3: loss linf 1.239 l1 1.104  cos 0.778  conflict 0%  linf weaker 100%  (245 s)
epoch 4: loss linf 1.211 l1 1.062  cos 0.768  conflict 0%  linf weaker 100%  (327 s)
epoch 5: loss linf 1.191 l1 1.020  cos 0.756  conflict 0%  linf weaker 100%  (409 s)
epoch 6: loss linf 1.159 l1 0.986  cos 0.750  conflict 0%  linf weaker 100%  (491 s)
epoch 7: loss linf 1.137 l1 0.955  cos 0.740  conflict 0%  linf weaker 100%  (573 s)
epoch 8: loss linf 1.121 l1 0.938  cos 0.738  conflict 0%  linf weaker 100%  (654 s)
epoch 9: loss linf 1.111 l1 0.921  cos 0.731  conflict 0%  linf weaker 100%  (736 s)
epoch 10: loss linf 1.099 l1 0.913  cos 0.728  conflict 0%  linf weaker 100%  (817 s)
clean 77.9  linf 41.6  l2 62.1  l1 50.3  union 41.0   (123 s)


{'clean': 77.9,
 'linf': 41.6,
 'l2': 62.1,
 'l1': 50.3,
 'union': 41.0,
 'sec_per_epoch': 81.72709856033325}

Part (c) results table: base model, RAMP and my method, per seed and the mean of the two seeds.

In [24]:
def seed_mean(prefix):
    return pd.DataFrame([results[f"{prefix}_s0"], results[f"{prefix}_s1"]]).mean()


table = pd.DataFrame({
    "base": results["base"],
    "ramp_s0": results["ramp_s0"],
    "ramp_s1": results["ramp_s1"],
    "ramp_mean": seed_mean("ramp"),
    "mine_s0": results["mine_s0"],
    "mine_s1": results["mine_s1"],
    "mine_mean": seed_mean("mine"),
}).T[COLS]
display(table.round(1))

,clean,linf,l2,l1,union,sec_per_epoch
base,78.7,48.4,59.8,22.4,22.3,NaN
ramp_s0,79.2,40.7,61.6,44.7,39.1,87.2
ramp_s1,79.0,42.1,62.7,46.8,40.6,87.1
ramp_mean,79.1,41.4,62.2,45.8,39.8,87.2
mine_s0,77.9,40.7,61.8,48.2,39.4,81.7
mine_s1,77.9,41.6,62.1,50.3,41.0,81.7
mine_mean,77.9,41.2,62.0,49.2,40.2,81.7


Ablation: remove the surgery (`sum`, the plain average of the two norm gradients), seeds 0 and 1.

In [ ]:
run("abl_sum_s0", lambda m: adv_train_wnf(m, my_loss, seed=0, mode="sum"))

epoch 1: loss linf 1.287 l1 1.313  cos 0.778  conflict 0%  linf weaker 67%  (82 s)
epoch 2: loss linf 1.259 l1 1.156  cos 0.795  conflict 0%  linf weaker 100%  (163 s)
epoch 3: loss linf 1.242 l1 1.107  cos 0.784  conflict 0%  linf weaker 100%  (245 s)
epoch 4: loss linf 1.221 l1 1.066  cos 0.767  conflict 0%  linf weaker 100%  (327 s)
epoch 5: loss linf 1.193 l1 1.030  cos 0.756  conflict 0%  linf weaker 100%  (409 s)
epoch 6: loss linf 1.166 l1 0.994  cos 0.744  conflict 0%  linf weaker 100%  (491 s)
epoch 7: loss linf 1.137 l1 0.956  cos 0.735  conflict 0%  linf weaker 100%  (572 s)
epoch 8: loss linf 1.119 l1 0.938  cos 0.728  conflict 0%  linf weaker 100%  (654 s)
epoch 9: loss linf 1.102 l1 0.923  cos 0.733  conflict 0%  linf weaker 100%  (736 s)


In [27]:
run("abl_sum_s1", lambda m: adv_train_wnf(m, my_loss, seed=1, mode="sum"))

epoch 1: loss linf 1.285 l1 1.317  cos 0.777  conflict 0%  linf weaker 65%  (82 s)
epoch 2: loss linf 1.262 l1 1.161  cos 0.792  conflict 0%  linf weaker 100%  (164 s)
epoch 3: loss linf 1.238 l1 1.104  cos 0.776  conflict 0%  linf weaker 100%  (246 s)
epoch 4: loss linf 1.211 l1 1.061  cos 0.767  conflict 0%  linf weaker 100%  (327 s)
epoch 5: loss linf 1.191 l1 1.020  cos 0.754  conflict 0%  linf weaker 100%  (409 s)
epoch 6: loss linf 1.160 l1 0.986  cos 0.749  conflict 0%  linf weaker 100%  (491 s)
epoch 7: loss linf 1.138 l1 0.955  cos 0.738  conflict 0%  linf weaker 100%  (572 s)
epoch 8: loss linf 1.121 l1 0.938  cos 0.736  conflict 0%  linf weaker 100%  (654 s)
epoch 9: loss linf 1.111 l1 0.921  cos 0.729  conflict 0%  linf weaker 100%  (736 s)
epoch 10: loss linf 1.099 l1 0.913  cos 0.729  conflict 0%  linf weaker 100%  (817 s)
clean 78.2  linf 41.8  l2 62.5  l1 50.4  union 41.0   (123 s)


{'clean': 78.2,
 'linf': 41.8,
 'l2': 62.5,
 'l1': 50.4,
 'union': 41.0,
 'sec_per_epoch': 81.72218918800354}

Optional second ablation: symmetric surgery (`pcgrad`) instead of weakest-norm-first, seed 0.

In [28]:
run("abl_pcgrad_s0", lambda m: adv_train_wnf(m, my_loss, seed=0, mode="pcgrad"))

epoch 1: loss linf 1.287 l1 1.314  cos 0.779  conflict 0%  linf weaker 68%  (82 s)
epoch 2: loss linf 1.258 l1 1.156  cos 0.796  conflict 0%  linf weaker 100%  (163 s)
epoch 3: loss linf 1.241 l1 1.108  cos 0.784  conflict 0%  linf weaker 100%  (245 s)
epoch 4: loss linf 1.221 l1 1.066  cos 0.765  conflict 0%  linf weaker 100%  (326 s)
epoch 5: loss linf 1.193 l1 1.030  cos 0.755  conflict 0%  linf weaker 100%  (408 s)
epoch 6: loss linf 1.166 l1 0.993  cos 0.744  conflict 0%  linf weaker 100%  (490 s)
epoch 7: loss linf 1.137 l1 0.957  cos 0.737  conflict 0%  linf weaker 100%  (572 s)
epoch 8: loss linf 1.119 l1 0.939  cos 0.729  conflict 0%  linf weaker 100%  (653 s)
epoch 9: loss linf 1.103 l1 0.923  cos 0.733  conflict 0%  linf weaker 100%  (735 s)
epoch 10: loss linf 1.097 l1 0.917  cos 0.726  conflict 0%  linf weaker 100%  (816 s)
clean 78.2  linf 40.8  l2 61.6  l1 48.5  union 39.7   (123 s)


{'clean': 78.2,
 'linf': 40.8,
 'l2': 61.6,
 'l1': 48.5,
 'union': 39.7,
 'sec_per_epoch': 81.64981083869934}

Ablation table, means over seeds 0 and 1 (the `pcgrad` row is seed 0 only).

In [29]:
ablation = pd.DataFrame({
    "mine (wnf)": seed_mean("mine"),
    "no surgery (sum)": seed_mean("abl_sum"),
    "pcgrad (seed 0)": results.get("abl_pcgrad_s0", {}),
}).T[COLS]
display(ablation.round(1))

,clean,linf,l2,l1,union,sec_per_epoch
mine (wnf),77.9,41.2,62.0,49.2,40.2,81.7
no surgery (sum),78.1,41.2,62.0,49.4,40.2,81.8
pcgrad (seed 0),78.2,40.8,61.6,48.5,39.7,81.6


Gradient statistics per epoch for my method (seed 0) and the no-surgery ablation (seed 0).

In [30]:
for name in ["mine_s0", "abl_sum_s0"]:
    print(name)
    display(pd.DataFrame(histories[name]).round(3))

mine_s0


,loss_linf,loss_l1,linf_weak,cos,conflict,seconds
0,1.287,1.314,0.667,0.778,0.0,81.679
1,1.259,1.156,1.000,0.796,0.0,163.565
2,1.242,1.108,1.000,0.783,0.0,245.231
3,1.221,1.066,1.000,0.765,0.0,326.747
4,1.193,1.029,1.000,0.754,0.0,408.296
5,1.165,0.993,1.000,0.744,0.0,489.959
6,1.137,0.957,1.000,0.735,0.0,571.653
7,1.119,0.938,1.000,0.726,0.0,653.382
8,1.103,0.922,1.000,0.732,0.0,735.187
9,1.098,0.917,1.000,0.723,0.0,817.021


abl_sum_s0


,loss_linf,loss_l1,linf_weak,cos,conflict,seconds
0,1.287,1.313,0.667,0.778,0.0,81.707
1,1.259,1.156,1.000,0.795,0.0,163.492
2,1.242,1.107,1.000,0.784,0.0,245.280
3,1.221,1.066,1.000,0.767,0.0,327.087
4,1.193,1.030,1.000,0.756,0.0,408.901
5,1.166,0.994,1.000,0.744,0.0,490.680
6,1.137,0.956,1.000,0.735,0.0,572.479
7,1.119,0.938,1.000,0.728,0.0,654.279
8,1.102,0.923,1.000,0.733,0.0,736.040
9,1.097,0.916,1.000,0.726,0.0,817.795


Check for gradient masking: evaluate my best model again with `steps=200, restarts=5`.

In [31]:
best = "mine_s0" if results["mine_s0"]["union"] >= results["mine_s1"]["union"] else "mine_s1"
print("best:", best)
strong = test_model(models[best], steps=200, restarts=5)
display(pd.DataFrame({"default (50 steps, 2 restarts)": results[best], "strong (200 steps, 5 restarts)": strong}).T[COLS[:5]].round(1))

best: mine_s1
clean 77.9  linf 41.5  l2 61.9  l1 48.0  union 40.3   (1216 s)


,clean,linf,l2,l1,union
"default (50 steps, 2 restarts)",77.9,41.6,62.1,50.3,41.0
"strong (200 steps, 5 restarts)",77.9,41.5,61.9,48.0,40.3


Optional: the same strong evaluation for RAMP seed 0.

In [32]:
strong_ramp = test_model(models["ramp_s0"], steps=200, restarts=5)
display(pd.DataFrame({"default": results["ramp_s0"], "strong": strong_ramp}).T[COLS[:5]].round(1))

clean 79.2  linf 40.6  l2 61.6  l1 43.0  union 38.0   (1216 s)


,clean,linf,l2,l1,union
default,79.2,40.7,61.6,44.7,39.1
strong,79.2,40.6,61.6,43.0,38.0
